# Hebrew Acronym Disambiguation: Local Dev Study

We compare **free expansion** and **candidate selection** using the same Qwen2.5 7B model with the sentence, alongside Ben’s fine-tuned DictaBERT. All systems receive the same identified occurrences; `[ACR]...[/ACR]` marks the exact target. Generation receives no candidate inventory. This comparison does not estimate the contribution of context.

The default is an invented preview with no model loading, service requests, research reads or saved outputs. Raw outputs remain unchanged; selection metrics are derived for development analysis. No research run or checkpoint validation was performed while preparing this notebook.

## Setup and manual verification

Run from this checkout in an isolated **local** Python environment. Dependencies are pinned in [requirements.txt](../requirements.txt); the checkpoint additionally requires its recorded `torch`, `transformers` and `tokenizers` versions and matching base-model/tokenizer content. Do not change the original checkpoint JSON to hide an incompatibility.

```sh
python3.12 -m venv .venv-study
.venv-study/bin/python -m pip install -e . jupyterlab ipykernel
.venv-study/bin/python -m pip check
.venv-study/bin/python -m jupyter lab notebooks/experimental_study.ipynb
```

Select that environment’s kernel. Configure explicit weight and optional relocated snapshot paths below; `inspect_checkpoint` checks local identities without loading the model. The weights and adjacent `<weights>.json` must be Ben’s original pair. `snapshot_path` permits a relocated identical snapshot after content verification. CPU is the default; choose `mps` or `cuda` explicitly only if supported. Colab remains an alternative with an installed checkout and compatible artifacts.

For Qwen, the selected connection is local Ollama with **`qwen2.5:7b`**. The user can run `ollama list` and `curl http://localhost:11434/api/tags` to check an already provisioned service/model. The notebook never starts the service or pulls models. Its enabled connection reads the actual server version, installed digest and model defaults, and verifies the digest around each bounded request. No manually supplied revision is called verified. Unspecified server defaults remain explicitly unspecified.

1. Fill `checkpoint` (and `snapshot_path` if relocated), set `mode="run"`, keep `run_kind="validation"`, and enable the encoder. This predicts only the first three qualified dev items in a fresh run directory.
2. To verify Qwen, enable it in that same new validation run: **one request per LLM mode**; other selected items remain visibly unrun. Encoder-only and reload work without Qwen configuration. Do not keep another model resident on the same GPU during encoder inference.
3. After manual success, restart from the settings cell with a fresh run ID and `run_kind="full_dev"` to request all 62 dev items per enabled system. Validation artifacts are never presented as full-dev results.
4. For existing output, use `mode="reload"`, `saved_run` and its exact `saved_run_id`. Reload never predicts or rewrites raw outputs. Saved encoder reuse instead requires `saved_encoder` and `saved_encoder_run_id`, identical input rows and checkpoint provenance.

Run all cells in order after changing settings. The output root is outside Git; no training is included.

In [ ]:
from pathlib import Path
from uuid import uuid4
from IPython.display import HTML, display
from hebrew_acronyms import experimental_study as study
from hebrew_acronyms.models.dictabert_cross_encoder.model import inspect_checkpoint

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
settings = dict(
    mode="preview", run_kind="validation", run_id=str(uuid4()), root=ROOT,
    input_path=ROOT / "data/study_v1/encoder_inputs/dev.csv", expected_dev_items=62,
    validation_items=3, device="cpu", checkpoint=None, snapshot_path=None,
    enable_encoder=False, saved_encoder=None, saved_encoder_run_id=None,
    enable_llm=False, qwen_model="qwen2.5:7b", ollama_url="http://localhost:11434",
    request_timeout=120, qwen_options={},
    output_root=ROOT.parent / "artifacts/study-runs", saved_run=None, saved_run_id=None,
)
checkpoint_info = None
if settings["mode"] == "run" and settings["enable_encoder"]:
    checkpoint_info = inspect_checkpoint(settings["checkpoint"], snapshot_path=settings["snapshot_path"])
checkpoint_info

## 1. Qualified dev input and run identity

The authorized dev file contains 62 items. Validation uses a clearly labelled prefix subset of that file; preview uses invented examples. Input identity binds complete rows, order and item IDs. Every selected item receives a record for each system, including disabled, unfinished and failed attempts. Historical dev/test or mixed review files are never chosen as fallbacks.

In [ ]:
rows, cohort = study.read_study_inputs(settings)
artifact = study.prepare_study(rows, cohort, settings)
run = artifact["settings"]  # Frozen settings used by the following source calls
print("Run:", artifact["run_id"], "Cohort:", cohort)

## 2. DictaBERT predictions

The existing loader verifies the original weights, model/tokenizer content and library identities. A verified local snapshot relocation changes only the load path. The existing evaluator returns one candidate prediction or failure per item. The source helper releases model memory before the Qwen steps; it never retrains.

In [ ]:
encoder_predictions = None
if settings["mode"] == "run" and (run["enable_encoder"] or run["saved_encoder"]):
    encoder_predictions = study.predict_encoder(
        artifact, rows, checkpoint=run["checkpoint"], snapshot_path=run["snapshot_path"],
        device=run["device"], saved_path=run["saved_encoder"], saved_run_id=run["saved_encoder_run_id"],
    )

## 3. Free expansion with the marked sentence

The raw target includes any attached prefix. Its exact span is marked in a separate prompt sentence; the original sentence remains unchanged. Generation receives no gold answer or candidate inventory. Outputs are retained for human review, including valid alternate phrasings; no substring score is used.

In [ ]:
generation_records = None
if settings["mode"] == "run" and run["enable_llm"]:
    qwen = study.connect_qwen(artifact)
    request_limit = 1 if run["run_kind"] == "validation" else None
    generation_records = study.collect_responses(
        artifact, "generate", qwen, limit=request_limit, on_record=study.save_study,
    )

## 4. Candidate selection with the same model

Selection uses the same marked sentence and bound Qwen callable. Candidate order follows the existing shared shuffle seed and is saved for every request. Only a single **uppercase** in-range letter after trimming is accepted: `A` can map to a candidate; `a`, `A or B`, extra explanation, an out-of-range letter and an empty response are parse failures. The raw response is always retained. A singleton is valid; inventories exceeding the 26-letter prompt capacity fail explicitly.

In [ ]:
selection_records = None
if settings["mode"] == "run" and run["enable_llm"]:
    selection_records = study.collect_responses(
        artifact, "select", qwen, limit=request_limit, on_record=study.save_study,
    )

## 5. Save raw evidence

Each LLM attempt is saved as it finishes, including service failures and interrupted requests. A digest mismatch retains any received answer with an identity failure status. All stages share one input/run identity. Re-executing a completed prediction stage is rejected; start a fresh run instead. Reload validates stored identity and consistency, independently of later prompt-code changes.

In [ ]:
artifact_path = None
if settings["mode"] == "run":
    artifact_path = study.save_study(artifact)
print("Saved artifact:", artifact_path)

## 6. Inspect items, selection accuracy and disagreements

The shared evaluator matches selected candidates to gold using **exact equality after trimming**, without aliases, quote folding or substrings. **Micro accuracy** counts correct selections over all requested items. **Macro accuracy** is the unweighted mean of within-`type_id` accuracies. Prediction/parse failures and unrun items remain in both denominators. An unfinished system and a validation subset are explicitly labelled; neither is silently presented as a completed full-dev result. Missing type metadata makes macro unavailable rather than inventing groups.

The table joins original context, target/span, gold, encoder prediction, generation answer, selection answer, displayed letter mapping, decoded choice and failures. Disagreements include only pairs of valid selections; failures remain visible separately. Generation is for manual review and has no automatic accuracy. These are preliminary dev measures, not test results or a final generation judgment protocol.

In [ ]:
prediction_table, results = study.display_tables(artifact)
display(HTML(prediction_table))
metrics = results["metrics"]
disagreements = results["disagreements"]

### Limits and reproducibility

Inspect `checkpoint_info`, `rows`, `artifact`, `encoder_predictions`, `metrics` and `disagreements` to follow each stage. Exact prompts, responses, shown orders, source hashes, checkpoint origin and server-reported model/settings evidence remain in the raw artifact. Old artifacts lacking newer provenance fields stay readable with explicit limitations. A prompt change does not invalidate historical viewing, but it changes the procedure for new runs.

Fixture tests establish code behavior, not compatibility or quality of Ben’s actual checkpoint or a live Ollama service. The first manual validation run supplies that evidence. The [training appendix](train_dictabert.ipynb) remains separate. AI assistance contributed implementation and fixture checks, not human annotation or scientific validation.